# 01 · Train the Attention-UNet

What this notebook does:
* trains the model described in `configs/base.yaml` on LOL-v1 (485 training pairs),
* saves `latest.pth` every epoch and `best.pth` (best validation PSNR) to
  `MyDrive/lowlight-attn-unet/runs/<experiment>/`,
* **resumes automatically** if Colab disconnects. Just run the whole notebook again.

The full run is 300 epochs (about 145k iterations). On a free T4 that usually takes more than one session, so `MAX_HOURS` stops training cleanly before Colab's limit. Re-run the notebook to continue where it stopped.

`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

## Setup
Runs in about 1–2 minutes. It mounts Google Drive, gets the latest code, installs the requirements, and restores the datasets from the zip cache on your Drive. Run `00_setup_and_data.ipynb` once beforehand.

In [ ]:
REPO_URL = "https://github.com/Asad3404/LUMA.git"
BRANCH = "claude/attention-unet-low-light-ay0gpv"   # change to "main" once merged
DRIVE_ROOT = "/content/drive/MyDrive/lowlight-attn-unet"
DATA_ROOT = "/content/data"
REPO_DIR = "/content/LUMA"

from google.colab import drive
drive.mount("/content/drive")
import os
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="code version: %h  %s"
!pip install -q -r requirements.txt 
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!python data/download.py --datasets lol_v1 --data_root {DATA_ROOT} --cache_dir {DRIVE_ROOT}/datasets --results_dir {DRIVE_ROOT}/results/data

## Settings
`CONFIG` picks what to train:
* `configs/base.yaml`: the main model, trained on LOL-v1 (do this first)
* `configs/lol_v2_real.yaml` / `configs/lol_v2_synthetic.yaml`: the same model trained on LOL-v2, for fair in-domain LOL-v2 numbers (optional; download those datasets in 00 first)

In [ ]:
CONFIG = "configs/base.yaml"
MAX_HOURS = 2.6     # stop cleanly before the free-tier session ends; re-run to resume
EXTRA_OPTS = ""     # e.g. "train.epochs=10" for a quick test run

## Model size
Prints the number of parameters and the multiply-accumulates (MACs) per image.

In [ ]:
!python -m src.models.unet_attn --config {CONFIG} | tail -3

## TensorBoard (live curves)
If the panel stays empty, wait for the first validation (every 5 epochs) and press the refresh button inside it.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_ROOT}/runs

## Train
Prints one line per epoch. `*best*` marks a new best validation PSNR. If the session dies, run the notebook again and it resumes from `latest.pth`.

In [ ]:
opts = f"--opts {EXTRA_OPTS}" if EXTRA_OPTS else ""
!python src/train.py --config {CONFIG} --max_hours {MAX_HOURS} {opts}

## Look at the run

In [ ]:
import json, glob
import matplotlib.pyplot as plt
from IPython.display import Image, display
import sys; sys.path.insert(0, ".")
from src.utils import load_config
exp = load_config(CONFIG)["experiment"]
run = f"{DRIVE_ROOT}/runs/{exp}"
rows = [json.loads(l) for l in open(f"{run}/metrics.jsonl")]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([r["epoch"] for r in rows], [r["val_psnr"] for r in rows]); ax[0].set_title("validation PSNR (dB)"); ax[0].set_xlabel("epoch")
ax[1].plot([r["epoch"] for r in rows], [r["train_loss"] for r in rows]); ax[1].set_title("training loss"); ax[1].set_xlabel("epoch")
plt.show()
print(open(f"{run}/train_summary.json").read())
last = sorted(glob.glob(f"{run}/samples/*.jpg"))[-1]
print("columns: input | output | ground truth   ", last)
display(Image(last, width=900))

### Next
If `status` says `finished`, continue with `02_evaluate_and_benchmark.ipynb`. Otherwise run this notebook again to resume.